New Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import time
import os
import json
import random
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score

print("Mounting Google Drive...")
drive.mount('/content/drive')

BASE_PATH = '/content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/'
RATINGS_FILE = os.path.join(BASE_PATH, 'ratings.json')
METADATA_FILE = os.path.join(BASE_PATH, 'metadata.json')

# Path to the SAVED processed file
PROCESSED_PATH = '/content/drive/My Drive/resources/processed_for_colab/'
CHECKPOINT_FILE = os.path.join(PROCESSED_PATH, 'cleaned_movie_docs.parquet')

# Loading Sampled Ratings from Cell 1
SAMPLE_FRACTION = 0.20
ratings_data = []
print(f"Loading and sampling {RATINGS_FILE}...")
with open(RATINGS_FILE, 'r') as f:
    for line in f:
        if random.random() < SAMPLE_FRACTION:
            ratings_data.append(json.loads(line))
ratings_df = pd.DataFrame(ratings_data)
print(f"Loaded {len(ratings_data)} sample ratings.")

# Loading Metadata Cell 1
print(f"Loading {METADATA_FILE}...")
metadata_df = pd.read_json(METADATA_FILE, lines=True)
print("Loaded metadata.")

# Loading Cleaned Text File
print(f"Loading checkpointed text data from {CHECKPOINT_FILE}...")
cleaned_docs_df = pd.read_parquet(CHECKPOINT_FILE)
print("Loaded cleaned text data.")

print("\n--- All Data Loaded Successfully ---")
print(f"Ratings (Sampled): {ratings_df.shape}")
print(f"Cleaned Text: {cleaned_docs_df.shape}")
print(f"Metadata: {metadata_df.shape}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading and sampling /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/ratings.json...
Loaded 5699763 sample ratings.
Loading /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/metadata.json...
Loaded metadata.
Loading checkpointed text data from /content/drive/My Drive/resources/processed_for_colab/cleaned_movie_docs.parquet...
Loaded cleaned text data.

--- All Data Loaded Successfully ---
Ratings (Sampled): (5699763, 3)
Cleaned Text: (52081, 2)
Metadata: (84661, 7)


Environment setup and loading the data (First setup, use the new one)

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import time
import os
import json
import random
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

BASE_PATH = '/content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/'

if not os.path.exists(BASE_PATH):
    print(f"ERROR: Path not found: {BASE_PATH}")
else:
    print(f"Base path found: {BASE_PATH}")

RATINGS_FILE = os.path.join(BASE_PATH, 'ratings.json')
REVIEWS_FILE = os.path.join(BASE_PATH, 'reviews.json')
METADATA_FILE = os.path.join(BASE_PATH, 'metadata.json')


# SAMPLING 20% of the file (full file crashes)
SAMPLE_FRACTION = 0.20  # load 20% of ratings
ratings_data = []
print(f"Loading and sampling {RATINGS_FILE} (keeping {SAMPLE_FRACTION*100}%)...")
start_time = time.time()

with open(RATINGS_FILE, 'r') as f:
    for line in f:
        if random.random() < SAMPLE_FRACTION:
            ratings_data.append(json.loads(line))

ratings_df = pd.DataFrame(ratings_data)
print(f"Loaded {len(ratings_data)} sample ratings in {time.time() - start_time:.2f} seconds")


print(f"Loading {REVIEWS_FILE}...")
start_time = time.time()
reviews_df = pd.read_json(REVIEWS_FILE, lines=True)
print(f"Loaded reviews_df in {time.time() - start_time:.2f} seconds")


print(f"Loading {METADATA_FILE}...")
start_time = time.time()
metadata_df = pd.read_json(METADATA_FILE, lines=True)
print(f"Loaded metadata_df in {time.time() - start_time:.2f} seconds")


print("\n--- Data Shapes ---")
print(f"Ratings (Sampled): {ratings_df.shape}")
print(f"Reviews: {reviews_df.shape}")
print(f"Metadata: {metadata_df.shape}")

print("\n--- Sampled Ratings Head ---")
print(ratings_df.head())

Mounting Google Drive...
Mounted at /content/drive
Base path found: /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/
Loading and sampling /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/ratings.json (keeping 20.0%)...
Loaded 5696776 sample ratings in 37.67 seconds
Loading /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/reviews.json...
Loaded reviews_df in 107.47 seconds
Loading /content/drive/My Drive/resources/genome_2021/movie_dataset_public_final/raw/metadata.json...
Loaded metadata_df in 1.55 seconds

--- Data Shapes ---
Ratings (Sampled): (5696776, 3)
Reviews: (2624608, 2)
Metadata: (84661, 7)

--- Sampled Ratings Head ---
   item_id  user_id  rating
0        5   997206     3.0
1       46   997206     4.0
2       85   997206     2.0
3      110   997206     5.0
4      204   997206     2.0


Data processing (First process, use the new one)

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Applies basic NLP preprocessing."""
    text = str(text).lower()  # converting to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # removing punctuation
    text = re.sub(r'\d+', '', text)  # removing numbers

    words = [word for word in text.split() if word not in stop_words]

    return ' '.join(words)

print("Aggregating all reviews for each movie...")
start_time = time.time()
movie_docs_series = reviews_df.groupby('item_id')['txt'].apply(' '.join)
print(f"Aggregated {len(reviews_df)} reviews into {len(movie_docs_series)} documents in {time.time() - start_time:.2f} sec")

del reviews_df

print(f"Cleaning {len(movie_docs_series)} movie documents...")
start_time = time.time()
cleaned_docs_series = movie_docs_series.apply(clean_text)
print(f"Cleaning completed in {time.time() - start_time:.2f} sec")

print("\n--- Checkpointing Data ---")

PROCESSED_PATH = '/content/drive/My Drive/resources/processed_for_colab/'
if not os.path.exists(PROCESSED_PATH):
    os.makedirs(PROCESSED_PATH)

CHECKPOINT_FILE = os.path.join(PROCESSED_PATH, 'cleaned_movie_docs.parquet')

cleaned_docs_df = pd.DataFrame(cleaned_docs_series).reset_index()
cleaned_docs_df.columns = ['item_id', 'cleaned_text']

# saving as Parquet for faster and smaller than CSV
print(f"Saving processed data to {CHECKPOINT_FILE}...")
cleaned_docs_df.to_parquet(CHECKPOINT_FILE, index=False)
print("Checkpoint successful!")

print("\n--- Processing Results (Sample) ---")
print(f"Original text for item_id {movie_docs_series.index[0]}:\n{movie_docs_series.iloc[0][:500]}...")
print("-" * 20)
print(f"Cleaned text for item_id {cleaned_docs_df['item_id'].iloc[0]}:\n{cleaned_docs_df['cleaned_text'].iloc[0][:500]}...")

NameError: name 'nltk' is not defined

Splitting data into Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

EVALUATION_SAMPLE_SIZE = 5000

# 80% train, 20% test
print(f"Splitting ratings_df (shape: {ratings_df.shape}) into 80/20 train/test sets...")
train_df, test_df = train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"  -> Train set shape: {train_df.shape}")
print(f"  -> Test set shape: {test_df.shape}")

print(f"\nCreating a test sample of {EVALUATION_SAMPLE_SIZE} ratings...")
test_sample_df = test_df.sample(
    n=EVALUATION_SAMPLE_SIZE,
    random_state=42
)

print(f"  -> test_sample_df shape: {test_sample_df.shape}")
print("\n--- Test Sample Head ---")
print(test_sample_df.head())

Splitting ratings_df (shape: (5699763, 3)) into 80/20 train/test sets...
  -> Train set shape: (4559810, 3)
  -> Test set shape: (1139953, 3)

Creating a test sample of 5000 ratings...
  -> test_sample_df shape: (5000, 3)

--- Test Sample Head ---
         item_id  user_id  rating
2610196    99750   965468     3.5
2513317     4034   476438     5.0
4207947     2080    56510     4.0
277215       471    99588     2.0
1188589     1264   912913     3.0


Baselines

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

results_summary = {}

# True Ratings
true_ratings = test_sample_df['rating']

print(f"Calculating baselines using {len(true_ratings)} sample ratings...")

print("\n--- 1. Random Baseline ---")
possible_ratings = np.arange(0.5, 5.5, 0.5)
random_predictions = np.random.choice(possible_ratings, size=len(true_ratings))

# MAE and RMSE
random_mae = mean_absolute_error(true_ratings, random_predictions)
random_rmse = np.sqrt(mean_squared_error(true_ratings, random_predictions))

print(f"  -> Random MAE: {random_mae:.4f}")
print(f"  -> Random RMSE: {random_rmse:.4f}")

results_summary['random_baseline'] = {
    "mae": random_mae,
    "rmse": random_rmse,
    "hit_ratio": np.nan
}

# Global Average Baseline
print("\n--- 2. Global Average Baseline ---")
global_avg_rating = train_df['rating'].mean()
print(f"Global average rating (from train set): {global_avg_rating:.4f}")

# creating 5000 predictions, all are the same avg value
avg_predictions = [global_avg_rating] * len(true_ratings)

# MAE and RMSE
avg_mae = mean_absolute_error(true_ratings, avg_predictions)
avg_rmse = np.sqrt(mean_squared_error(true_ratings, avg_predictions))

print(f"  -> Global Avg MAE: {avg_mae:.4f}")
print(f"  -> Global Avg RMSE: {avg_rmse:.4f}")

results_summary['avg_baseline'] = {
    "mae": avg_mae,
    "rmse": avg_rmse,
    "hit_ratio": np.nan
}

print("\n--- LIVE RESULTS SUMMARY ---")
print(pd.DataFrame.from_dict(results_summary, orient='index'))

Calculating baselines using 5000 sample ratings...

--- 1. Random Baseline ---
  -> Random MAE: 1.5861
  -> Random RMSE: 1.9565

--- 2. Global Average Baseline ---
Global average rating (from train set): 3.5292
  -> Global Avg MAE: 0.8355
  -> Global Avg RMSE: 1.0545

--- LIVE RESULTS SUMMARY ---
                     mae      rmse  hit_ratio
random_baseline  1.58610  1.956489        NaN
avg_baseline     0.83554  1.054528        NaN


TF-IDF and Cosine Similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import time

print("TF-IDF + On-the-Fly Cosine")
print("Vectorizing cleaned text with TF-IDF...")
start_time = time.time()

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=20000, min_df=5)
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_docs_df['cleaned_text'])

print(f"  -> TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  -> Vectorizing finished in {time.time() - start_time:.2f} sec")

print("Creating helper mappings for evaluation...")
movie_indices = pd.Series(
    cleaned_docs_df.index,
    index=cleaned_docs_df['item_id']
)
print("Grouping training data by user for fast lookup...")
train_user_groups = train_df.groupby('user_id')
global_avg_rating = train_df['rating'].mean()

# Prediction Function (On-the-fly Weighted Average)
def predict_rating_cosine_onthefly(user_id, item_id, train_groups, text_matrix, indices_map):

    default_prediction = global_avg_rating

    try:
        # get the vector for target movie
        target_item_vector = text_matrix[indices_map[item_id]]
    except KeyError:
        return default_prediction

    try:
        user_history = train_groups.get_group(user_id)
    except KeyError:
        return default_prediction

    # collect neighbor movie indices and their ratings
    neighbor_indices = []
    neighbor_ratings_list = []

    for _, row in user_history.iterrows():
        neighbor_item_id = row['item_id']
        try:
            neighbor_indices.append(indices_map[neighbor_item_id])
            neighbor_ratings_list.append(row['rating'])
        except KeyError:
            continue

    if not neighbor_indices:
        return default_prediction

    neighbor_vectors = text_matrix[neighbor_indices]

    # cosine similarity for ONLY the pairs needed
    sim_scores_list = cosine_similarity(target_item_vector, neighbor_vectors)[0]

    sim_scores = []
    neighbor_ratings = []

    for i, sim in enumerate(sim_scores_list):
        if sim > 0:
            sim_scores.append(sim)
            neighbor_ratings.append(neighbor_ratings_list[i])

    if not sim_scores:
        return default_prediction

    weighted_sum = np.dot(sim_scores, neighbor_ratings)
    sum_of_weights = np.sum(sim_scores)

    if sum_of_weights == 0:
        return default_prediction

    prediction = weighted_sum / sum_of_weights
    return prediction

# MAE / RMSE
print("\nTF-IDF aand Cosine Model")
print(f"Running prediction loop for {len(test_sample_df)} sample ratings...")
start_time = time.time()

predictions = []
for row in test_sample_df.itertuples():
    pred = predict_rating_cosine_onthefly(
        user_id=row.user_id,
        item_id=row.item_id,
        train_groups=train_user_groups,
        text_matrix=tfidf_matrix,
        indices_map=movie_indices
    )
    predictions.append(pred)

print(f"  -> Prediction loop finished in {time.time() - start_time:.2f} sec")

# Reports
true_ratings = test_sample_df['rating']

cosine_mae = mean_absolute_error(true_ratings, predictions)
cosine_rmse = np.sqrt(mean_squared_error(true_ratings, predictions))

print(f"  -> Cosine Model MAE: {cosine_mae:.4f}")
print(f"  -> Cosine Model RMSE: {cosine_rmse:.4f}")

results_summary['cosine_tfidf'] = {
    "mae": cosine_mae,
    "rmse": cosine_rmse,
    "hit_ratio": np.nan
}

print("\n--- LIVE RESULTS SUMMARY ---")
print(pd.DataFrame.from_dict(results_summary, orient='index'))

TF-IDF + On-the-Fly Cosine
Vectorizing cleaned text with TF-IDF...
  -> TF-IDF matrix shape: (52081, 20000)
  -> Vectorizing finished in 229.65 sec
Creating helper mappings for evaluation...
Grouping training data by user for fast lookup...

TF-IDF aand Cosine Model
Running prediction loop for 5000 sample ratings...
  -> Prediction loop finished in 85.19 sec
  -> Cosine Model MAE: 0.7607
  -> Cosine Model RMSE: 0.9855

--- LIVE RESULTS SUMMARY ---
                      mae      rmse  hit_ratio
random_baseline  1.586100  1.956489        NaN
avg_baseline     0.835540  1.054528        NaN
cosine_tfidf     0.760731  0.985474        NaN


Binary Counts abd Jaccard simmilarity

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score

print("\nBinary Counts + Jaccard Similarity")
print("Vectorizing cleaned text with Binary CountVectorizer...")
start_time = time.time()

binary_vectorizer = CountVectorizer(
    max_features=20000,
    min_df=5,
    binary=True
)
binary_matrix = binary_vectorizer.fit_transform(cleaned_docs_df['cleaned_text'])

print(f"  -> Binary matrix shape: {binary_matrix.shape}")
print(f"  -> Vectorizing finished in {time.time() - start_time:.2f} sec")

print("Converting binary matrix to dense array for loop...")
binary_vectors = binary_matrix.toarray()
print(f"  -> Dense vectors shape: {binary_vectors.shape}")

# Prediction Function (Jaccard Weighted Average)
def predict_rating_jaccard(user_id, item_id, train_groups, vectors, indices_map):

    default_prediction = global_avg_rating

    try:
        target_item_vector = vectors[indices_map[item_id]]
    except KeyError:
        return default_prediction

    try:
        user_history = train_groups.get_group(user_id)
    except KeyError:
        return default_prediction

    sim_scores = []
    neighbor_ratings = []

    for _, row in user_history.iterrows():
        neighbor_item_id = row['item_id']
        neighbor_rating = row['rating']

        try:
            neighbor_item_vector = vectors[indices_map[neighbor_item_id]]
        except KeyError:
            continue

        sim = jaccard_score(target_item_vector, neighbor_item_vector, average='binary')

        if sim > 0:
            sim_scores.append(sim)
            neighbor_ratings.append(neighbor_rating)

    if not sim_scores:
        return default_prediction

    weighted_sum = np.dot(sim_scores, neighbor_ratings)
    sum_of_weights = np.sum(sim_scores)

    if sum_of_weights == 0:
        return default_prediction

    prediction = weighted_sum / sum_of_weights
    return prediction

# MAE / RMSE
print("\nBinary and Jaccard Model")
print(f"Running prediction loop for {len(test_sample_df)} sample ratings...")
start_time = time.time()

predictions_jaccard = []
for row in test_sample_df.itertuples():
    pred = predict_rating_jaccard(
        user_id=row.user_id,
        item_id=row.item_id,
        train_groups=train_user_groups,
        vectors=binary_vectors,
        indices_map=movie_indices
    )
    predictions_jaccard.append(pred)

print(f"  -> Prediction loop finished in {time.time() - start_time:.2f} sec")

# Reports
jaccard_mae = mean_absolute_error(true_ratings, predictions_jaccard)
jaccard_rmse = np.sqrt(mean_squared_error(true_ratings, predictions_jaccard))

print(f"  -> Jaccard Model MAE: {jaccard_mae:.4f}")
print(f"  -> Jaccard Model RMSE: {jaccard_rmse:.4f}")

results_summary['jaccard_binary'] = {
    "mae": jaccard_mae,
    "rmse": jaccard_rmse,
    "hit_ratio": np.nan
}

print("\n--- LIVE RESULTS SUMMARY ---")
print(pd.DataFrame.from_dict(results_summary, orient='index'))


Binary Counts + Jaccard Similarity
Vectorizing cleaned text with Binary CountVectorizer...
  -> Binary matrix shape: (52081, 20000)
  -> Vectorizing finished in 229.95 sec
Converting binary matrix to dense array for loop...
  -> Dense vectors shape: (52081, 20000)

Binary and Jaccard Model
Running prediction loop for 5000 sample ratings...
  -> Prediction loop finished in 1087.61 sec
  -> Jaccard Model MAE: 0.7593
  -> Jaccard Model RMSE: 0.9850

--- LIVE RESULTS SUMMARY ---
                      mae      rmse  hit_ratio
random_baseline  1.586100  1.956489        NaN
avg_baseline     0.835540  1.054528        NaN
cosine_tfidf     0.760731  0.985474        NaN
jaccard_binary   0.759319  0.984968        NaN


TopN Recommendation and Hit Ratio

*TF-IDF Hit Ratio (on the fly new version)*

In [ ]:
print("TF-IDF Hit Ratio (On-the-fly)")
N = 10

hit_ratio_test_df = test_sample_df[test_sample_df['rating'] >= 4.0]
liked_train_df = train_df[train_df['rating'] >= 4.0]
liked_train_groups = liked_train_df.groupby('user_id')
print(f"Using {len(hit_ratio_test_df)} 'liked' items from the test sample.")

# TF-IDF evaluation loop
hits_tfidf = 0
start_time = time.time()
print("Running TF-IDF Hit Ratio loop...")
for row in hit_ratio_test_df.itertuples():
    user_id = row.user_id
    target_item_id = row.item_id

    try:
        user_history = liked_train_groups.get_group(user_id)
    except KeyError:
        continue

    liked_indices = []
    for item_id in user_history['item_id']:
        try:
            liked_indices.append(movie_indices[item_id])
        except KeyError:
            continue
    if not liked_indices:
        continue

    user_profile_vector = tfidf_matrix[liked_indices].mean(axis=0)

    try:
        target_item_index = movie_indices[target_item_id]
    except KeyError:
        continue

    # FIX! np.matrix to  np.array
    scores = cosine_similarity(np.asarray(user_profile_vector), tfidf_matrix)[0]

    top_n_indices = np.argpartition(scores, -N)[-N:]

    if target_item_index in top_n_indices:
        hits_tfidf += 1

# ReportS
cosine_hit_ratio = hits_tfidf / len(hit_ratio_test_df)
print(f"  -> TF-IDF loop finished in {time.time() - start_time:.2f} sec")
print(f"  -> TF-IDF Hits: {hits_tfidf} (out of {len(hit_ratio_test_df)})")
print(f"  -> TF-IDF Hit Ratio @ 10: {cosine_hit_ratio:.4f}")

results_summary['cosine_tfidf']['hit_ratio'] = cosine_hit_ratio
print("\nTF-IDF Hit Ratio Complete")

del liked_train_groups
del user_profile_vector
del liked_train_df
del hit_ratio_test_df
print("Cleared temp dataframes from memory.")

TF-IDF Hit Ratio (On-the-fly)
Using 2488 'liked' items from the test sample.
Running TF-IDF Hit Ratio loop...
  -> TF-IDF loop finished in 1565.68 sec
  -> TF-IDF Hits: 5 (out of 2488)
  -> TF-IDF Hit Ratio @ 10: 0.0020

TF-IDF Hit Ratio Complete
Cleared temp dataframes from memory.


*Binary/Cosine Hit Ratio (on the fly again, previous keeps crashes)*

In [ ]:
print("\nBinary/Cosine Hit Ratio (On-the-fly)")
hit_ratio_test_df = test_sample_df[test_sample_df['rating'] >= 4.0]
liked_train_df = train_df[train_df['rating'] >= 4.0]
liked_train_groups = liked_train_df.groupby('user_id')
print(f"Using {len(hit_ratio_test_df)} 'liked' items.")

hits_binary = 0
start_time = time.time()
print("Running Binary Hit Ratio loop...")
for row in hit_ratio_test_df.itertuples():
    user_id = row.user_id
    target_item_id = row.item_id

    try:
        user_history = liked_train_groups.get_group(user_id)
    except KeyError:
        continue

    liked_indices = []
    for item_id in user_history['item_id']:
        try:
            liked_indices.append(movie_indices[item_id])
        except KeyError:
            continue
    if not liked_indices:
        continue

    user_profile_vector = binary_vectors[liked_indices].mean(axis=0)

    try:
        target_item_index = movie_indices[target_item_id]
    except KeyError:
        continue

    scores = cosine_similarity(user_profile_vector.reshape(1, -1), binary_vectors)[0]
    top_n_indices = np.argpartition(scores, -N)[-N:]

    if target_item_index in top_n_indices:
        hits_binary += 1

binary_hit_ratio = hits_binary / len(hit_ratio_test_df)
print(f"  -> Binary loop finished in {time.time() - start_time:.2f} sec")
print(f"  -> Binary/Cosine Hits: {hits_binary} (out of {len(hit_ratio_test_df)})")
print(f"  -> Binary/Cosine Hit Ratio @ 10: {binary_hit_ratio:.4f}")

results_summary['jaccard_binary']['hit_ratio'] = binary_hit_ratio
print("\nBinary Hit Ratio Complete")

del liked_train_groups
del user_profile_vector
del liked_train_df
del hit_ratio_test_df
print("Cleared temp dataframes from memory.")


Binary/Cosine Hit Ratio (On-the-fly)
Using 2488 'liked' items.
Running Binary Hit Ratio loop...


KeyboardInterrupt: 

Final Report

In [ ]:
import json

print("FINAL RESULTS SUMMARY")
final_results_df = pd.DataFrame.from_dict(results_summary, orient='index')
final_results_df.index.name = "Model"

print(final_results_df)

print("\nFINAL RESULTS")
print(final_results_df.reset_index().to_markdown(index=False))

RESULTS_FILE = os.path.join(PROCESSED_PATH, 'final_results_summary.json')
print(f"\nSaving final results dictionary to {RESULTS_FILE}...")
with open(RESULTS_FILE, 'w') as f:
    json.dump(results_summary, f, indent=4)

print("Project complete.")

FINAL RESULTS SUMMARY
                      mae      rmse  hit_ratio
Model                                         
random_baseline  1.586100  1.956489        NaN
avg_baseline     0.835540  1.054528        NaN
cosine_tfidf     0.760731  0.985474    0.00201
jaccard_binary   0.759319  0.984968        NaN

FINAL RESULTS
| Model           |      mae |     rmse |    hit_ratio |
|:----------------|---------:|---------:|-------------:|
| random_baseline | 1.5861   | 1.95649  | nan          |
| avg_baseline    | 0.83554  | 1.05453  | nan          |
| cosine_tfidf    | 0.760731 | 0.985474 |   0.00200965 |
| jaccard_binary  | 0.759319 | 0.984968 | nan          |

Saving final results dictionary to /content/drive/My Drive/resources/processed_for_colab/final_results_summary.json...
Project complete.
